# Public Uptime Monitoring Detection

This notebook checks each platform URL from the NIH IC Efforts Landscape spreadsheet for signs of public uptime monitoring:

1. **Status page links** — Scans the homepage HTML for links to known status page services (Statuspage.io, UptimeRobot, Upptime, Cachet, Instatus, etc.) or subdomain patterns like `status.*`
2. **Health endpoints** — Probes common health/status API paths (`/status`, `/health`, `/healthcheck`, `/api/status`, etc.)
3. **HTTP headers** — Checks response headers for monitoring-related indicators

In [1]:
import openpyxl
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import re
import time
import warnings
warnings.filterwarnings('ignore')

xlsx_path = r'NIH IC Efforts Landscape 2026.04.22.xlsx'
wb = openpyxl.load_workbook(xlsx_path, data_only=True)
ws = wb['Ecosystems']

headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
rows = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    rows.append(list(row))

df = pd.DataFrame(rows, columns=headers)
df = df.dropna(subset=['Name'])

print(f"Loaded {len(df)} platforms to check.")
df[['Name', 'URL']].head(10)

Loaded 29 platforms to check.


,Name,URL
0,HEAL Data Platform,https://healdata.org/landing
1,Cancer Research Data Commons,https://datacommons.cancer.gov/
2,dkNET,https://dknet.org/
3,NAHDAP,https://www.icpsr.umich.edu/sites/nahdap/home
4,NICHD DASH,https://dash.nichd.nih.gov/
5,NIDA Data Share,https://datashare.nida.nih.gov/data
6,NIDDK Central Repository,https://repository.niddk.nih.gov/home
7,NIMH Data Archive,https://nda.nih.gov/
8,NLM Data Discovery,https://datadiscovery.nlm.nih.gov/
9,AnVIL (workspace),https://anvilproject.org/


## Detection Functions

In [2]:
REQUEST_TIMEOUT = 15
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT})

STATUS_PAGE_SERVICES = [
    'statuspage.io', 'status.io', 'uptimerobot.com', 'upptime.js.org',
    'instatus.com', 'cachet', 'betteruptime.com', 'openstatus.dev',
    'hyperping.io', 'freshstatus.io', 'pagerduty.com', 'opsgenie.com',
    'healthchecks.io', 'statuscake.com',
]

STATUS_LINK_PATTERNS = re.compile(
    r'status|uptime|system.health|incident|operational',
    re.IGNORECASE
)

HEALTH_ENDPOINTS = [
    '/status', '/health', '/healthcheck', '/api/status',
    '/api/health', '/api/v1/status', '/_health', '/heartbeat',
    '/ping', '/up',
]

MONITORING_HEADERS = [
    'x-uptime', 'x-health', 'x-statuspage', 'x-monitoring',
    'x-pingback', 'x-powered-by-statuspage',
]


def normalize_url(url):
    url = url.strip()
    if not url.startswith('http'):
        url = 'https://' + url
    return url


def check_status_page_links(url):
    """Scan homepage HTML for links to status pages or monitoring services."""
    findings = []
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, verify=False, allow_redirects=True)
        soup = BeautifulSoup(resp.text, 'html.parser')

        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].lower()
            text = a_tag.get_text(strip=True).lower()
            raw_href = a_tag['href']

            for svc in STATUS_PAGE_SERVICES:
                if svc in href:
                    findings.append({
                        'description': f"Status page link (service: {svc})",
                        'detected_url': raw_href,
                    })

            if STATUS_LINK_PATTERNS.search(href) or STATUS_LINK_PATTERNS.search(text):
                parsed = urlparse(raw_href)
                if parsed.hostname and 'status' in parsed.hostname:
                    findings.append({
                        'description': 'Status subdomain link',
                        'detected_url': raw_href,
                    })
                elif 'status' in text or 'uptime' in text:
                    already = [f['detected_url'] for f in findings]
                    if raw_href not in already:
                        findings.append({
                            'description': f"Possible status link (text: '{a_tag.get_text(strip=True)}')",
                            'detected_url': raw_href,
                        })

    except Exception as e:
        findings.append({'description': f"[Error fetching homepage: {type(e).__name__}]", 'detected_url': ''})
    return findings


def check_health_endpoints(url):
    """Probe common health/status endpoints."""
    findings = []
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    for endpoint in HEALTH_ENDPOINTS:
        full_url = base + endpoint
        try:
            resp = SESSION.get(full_url, timeout=8, verify=False, allow_redirects=True)
            if resp.status_code == 200:
                content_type = resp.headers.get('content-type', '')
                body_preview = resp.text[:200].strip()
                is_json = 'json' in content_type
                has_status_keyword = bool(re.search(
                    r'"status"|"healthy"|"ok"|"alive"|"up"|"operational"',
                    body_preview, re.IGNORECASE
                ))

                if is_json or has_status_keyword:
                    findings.append({
                        'description': f"Health endpoint: {endpoint} (200 OK, {'JSON' if is_json else 'HTML'}) — {body_preview[:80]}",
                        'detected_url': full_url,
                    })
                elif len(body_preview) < 500:
                    findings.append({
                        'description': f"Possible health endpoint: {endpoint} (200 OK) — {body_preview[:80]}",
                        'detected_url': full_url,
                    })
        except Exception:
            pass
    return findings


def check_monitoring_headers(url):
    """Check HTTP response headers for monitoring indicators."""
    findings = []
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, verify=False, allow_redirects=True)
        for header_name in MONITORING_HEADERS:
            for resp_header in resp.headers:
                if header_name in resp_header.lower():
                    findings.append({
                        'description': f"Header: {resp_header}: {resp.headers[resp_header]}",
                        'detected_url': url,
                    })
    except Exception as e:
        findings.append({'description': f"[Error checking headers: {type(e).__name__}]", 'detected_url': ''})
    return findings


def check_status_subdomain(url):
    """Check if a status.* subdomain exists for the site's domain."""
    findings = []
    parsed = urlparse(url)
    domain = parsed.netloc
    domain = re.sub(r'^(www\.)', '', domain)

    parts = domain.split('.')
    if len(parts) >= 2:
        root_domain = '.'.join(parts[-2:])
        status_url = f"https://status.{root_domain}"
        try:
            resp = SESSION.get(status_url, timeout=10, verify=False, allow_redirects=True)
            if resp.status_code == 200:
                findings.append({
                    'description': f"Status subdomain exists (200 OK)",
                    'detected_url': status_url,
                })
        except Exception:
            pass
    return findings


def check_site(name, url):
    """Run all checks on a single site."""
    url = normalize_url(url)
    print(f"  Checking {name}...", end=' ')

    results = {
        'status_page_links': check_status_page_links(url),
        'health_endpoints': check_health_endpoints(url),
        'monitoring_headers': check_monitoring_headers(url),
        'status_subdomain': check_status_subdomain(url),
    }

    total = sum(len([f for f in v if not f['description'].startswith('[Error')]) for v in results.values())
    print(f"found {total} indicator(s)")
    time.sleep(1)
    return results

print("Detection functions ready.")

Detection functions ready.


## Run Checks on All Platforms

This will take a few minutes as it checks each site with a 1-second delay between platforms.

In [3]:
print("Scanning all platforms for public uptime monitoring...\n")

all_results = {}
for _, row in df.iterrows():
    name = row['Name']
    url = row['URL']
    if pd.isna(url) or not url.strip():
        print(f"  Skipping {name} (no URL)")
        continue
    all_results[name] = check_site(name, url)

print(f"\nDone. Checked {len(all_results)} platforms.")

Scanning all platforms for public uptime monitoring...

  Checking HEAL Data Platform... found 0 indicator(s)
  Checking Cancer Research Data Commons... found 0 indicator(s)
  Checking dkNET... found 0 indicator(s)
  Checking NAHDAP... found 0 indicator(s)
  Checking NICHD DASH... found 0 indicator(s)
  Checking NIDA Data Share... found 0 indicator(s)
  Checking NIDDK Central Repository... found 1 indicator(s)
  Checking NIMH Data Archive... found 7 indicator(s)
  Checking NLM Data Discovery... found 0 indicator(s)
  Checking AnVIL (workspace)... found 0 indicator(s)
  Checking 4D Nucleome... found 0 indicator(s)
  Checking Common Fund Data Ecosystem Portal... found 10 indicator(s)
  Checking DataMed COVID-19... found 10 indicator(s)
  Checking DataMed/BioCaddie... found 0 indicator(s)
  Checking ExRNA... found 2 indicator(s)
  Checking Gabriella Miller Kids First... found 1 indicator(s)
  Checking HuBMAP... found 0 indicator(s)
  Checking Human Microbiome Project... found 0 indicator(

## Detailed Results

In [4]:
for name, results in all_results.items():
    has_findings = any(
        findings for findings in results.values()
        if findings and not all(f['description'].startswith('[Error') for f in findings)
    )
    if has_findings:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        for category, findings in results.items():
            if findings and not all(f['description'].startswith('[Error') for f in findings):
                label = category.replace('_', ' ').title()
                for f in findings:
                    if not f['description'].startswith('[Error'):
                        print(f"  [{label}] {f['description']}")
                        if f['detected_url']:
                            print(f"    -> {f['detected_url']}")

print("\n\nPlatforms with NO public uptime indicators detected:")
print("-" * 50)
for name, results in all_results.items():
    has_findings = any(
        findings for findings in results.values()
        if findings and not all(f['description'].startswith('[Error') for f in findings)
    )
    if not has_findings:
        print(f"  - {name}")


  NIDDK Central Repository
  [Health Endpoints] Possible health endpoint: /health (200 OK) — Healthy
    -> https://repository.niddk.nih.gov/health

  NIMH Data Archive
  [Health Endpoints] Possible health endpoint: /status (200 OK) — <!DOCTYPE html>
<html lang="en" data-beasties-container>
  <head>
    <meta char
    -> https://nda.nih.gov/status
  [Health Endpoints] Possible health endpoint: /health (200 OK) — <!DOCTYPE html>
<html lang="en" data-beasties-container>
  <head>
    <meta char
    -> https://nda.nih.gov/health
  [Health Endpoints] Possible health endpoint: /healthcheck (200 OK) — <!DOCTYPE html>
<html lang="en" data-beasties-container>
  <head>
    <meta char
    -> https://nda.nih.gov/healthcheck
  [Health Endpoints] Possible health endpoint: /_health (200 OK) — <!DOCTYPE html>
<html lang="en" data-beasties-container>
  <head>
    <meta char
    -> https://nda.nih.gov/_health
  [Health Endpoints] Possible health endpoint: /heartbeat (200 OK) — <!DOCTYPE html>
<html lan

## Summary Table

In [5]:
def has_real_findings(findings_list):
    return bool(findings_list) and not all(f['description'].startswith('[Error') for f in findings_list)

summary_rows = []
for name, results in all_results.items():
    url = df.loc[df['Name'] == name, 'URL'].values[0]
    summary_rows.append({
        'Name': name,
        'URL': url,
        'Status Page Links': has_real_findings(results['status_page_links']),
        'Health Endpoints': has_real_findings(results['health_endpoints']),
        'Monitoring Headers': has_real_findings(results['monitoring_headers']),
        'Status Subdomain': has_real_findings(results['status_subdomain']),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df['Any Detected'] = (
    summary_df['Status Page Links'] |
    summary_df['Health Endpoints'] |
    summary_df['Monitoring Headers'] |
    summary_df['Status Subdomain']
)

detected_count = summary_df['Any Detected'].sum()
print(f"Public uptime monitoring detected: {detected_count}/{len(summary_df)} platforms\n")

summary_df.style.applymap(
    lambda v: 'background-color: #d4edda' if v is True else
              ('background-color: #f8d7da' if v is False else ''),
    subset=['Status Page Links', 'Health Endpoints', 'Monitoring Headers',
            'Status Subdomain', 'Any Detected']
)

Public uptime monitoring detected: 10/29 platforms



,Name,URL,Status Page Links,Health Endpoints,Monitoring Headers,Status Subdomain,Any Detected
0,HEAL Data Platform,https://healdata.org/landing,False,False,False,False,False
1,Cancer Research Data Commons,https://datacommons.cancer.gov/,False,False,False,False,False
2,dkNET,https://dknet.org/,False,False,False,False,False
3,NAHDAP,https://www.icpsr.umich.edu/sites/nahdap/home,False,False,False,False,False
4,NICHD DASH,https://dash.nichd.nih.gov/,False,False,False,False,False
5,NIDA Data Share,https://datashare.nida.nih.gov/data,False,False,False,False,False
6,NIDDK Central Repository,https://repository.niddk.nih.gov/home,False,True,False,False,True
7,NIMH Data Archive,https://nda.nih.gov/,False,True,False,False,True
8,NLM Data Discovery,https://datadiscovery.nlm.nih.gov/,False,False,False,False,False
9,AnVIL (workspace),https://anvilproject.org/,False,False,False,False,False


## Export Results

In [6]:
import os
os.makedirs('results', exist_ok=True)

detail_rows = []
for name, results in all_results.items():
    url = df.loc[df['Name'] == name, 'URL'].values[0]
    for category, findings in results.items():
        for finding in findings:
            if not finding['description'].startswith('[Error'):
                detail_rows.append({
                    'Name': name,
                    'URL': url,
                    'Check Type': category.replace('_', ' ').title(),
                    'Finding': finding['description'],
                    'Detected URL': finding['detected_url'],
                })

if detail_rows:
    detail_df = pd.DataFrame(detail_rows)
    detail_df.to_csv('results/uptime_check_details.csv', index=False)
    print(f"Detailed findings saved to results/uptime_check_details.csv ({len(detail_df)} rows)")
    display(detail_df)
else:
    print("No findings to export.")

summary_df.to_csv('results/uptime_check_summary.csv', index=False)
print(f"\nSummary saved to results/uptime_check_summary.csv ({len(summary_df)} rows)")

Detailed findings saved to results/uptime_check_details.csv (55 rows)


,Name,URL,Check Type,Finding,Detected URL
0,NIDDK Central Repository,https://repository.niddk.nih.gov/home,Health Endpoints,Possible health endpoint: /health (200 OK) — H...,https://repository.niddk.nih.gov/health
1,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /status (200 OK) — <...,https://nda.nih.gov/status
2,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /health (200 OK) — <...,https://nda.nih.gov/health
3,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /healthcheck (200 OK...,https://nda.nih.gov/healthcheck
4,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /_health (200 OK) — ...,https://nda.nih.gov/_health
5,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /heartbeat (200 OK) ...,https://nda.nih.gov/heartbeat
6,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /ping (200 OK) — <!D...,https://nda.nih.gov/ping
7,NIMH Data Archive,https://nda.nih.gov/,Health Endpoints,Possible health endpoint: /up (200 OK) — <!DOC...,https://nda.nih.gov/up
8,Common Fund Data Ecosystem Portal,https://app.nih-cfde.org/,Health Endpoints,Possible health endpoint: /status (200 OK) — <...,https://app.nih-cfde.org/status
9,Common Fund Data Ecosystem Portal,https://app.nih-cfde.org/,Health Endpoints,Possible health endpoint: /health (200 OK) — <...,https://app.nih-cfde.org/health



Summary saved to results/uptime_check_summary.csv (29 rows)
